In [ ]:
%pip install gradio torch numpy librosa transformers typing dataclasses

In [3]:
import gradio as gr
import torch
import numpy as np
import librosa
import os
from transformers import Wav2Vec2BertModel, AutoFeatureExtractor, HubertModel
import torch.nn as nn
from typing import Optional, Tuple
from transformers.file_utils import ModelOutput
from dataclasses import dataclass
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss

C:\Users\Admin\Documents\ITMOPhD\LEARNING\MachineLearning\Experiment1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
@dataclass
class SpeechClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor]] = None
    attentions: Optional[Tuple[torch.FloatTensor]] = None

from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2PreTrainedModel,
    Wav2Vec2Model
)

In [5]:
class Wav2Vec2ClassificationHead(nn.Module):
    """Head for wav2vec classification task."""

    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = features
        x = self.dropout(x)
        x = self.dense(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

In [6]:
class Wav2Vec2ForSpeechClassification(nn.Module):
    def __init__(self,model_name):
        super().__init__()
        self.num_labels = 2
        self.wav2vec2bert = Wav2Vec2BertModel.from_pretrained(model_name)
        self.config = self.wav2vec2bert.config
        self.classifier = Wav2Vec2ClassificationHead(self.wav2vec2bert.config)

    def merge_strategy(self,hidden_states):
        return torch.mean(hidden_states, dim=1)

    def forward(self,input_features,attention_mask=None,output_attentions=None,output_hidden_states=None,return_dict=None,labels=None,):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.wav2vec2bert(
            input_features,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = outputs.last_hidden_state
        hidden_states = self.merge_strategy(hidden_states)
        logits = self.classifier(hidden_states)

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                self.config.problem_type = "multi_label_classification"

            loss_fct = BCEWithLogitsLoss()
            loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SpeechClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.last_hidden_state,
            attentions=outputs.attentions,
        )

In [7]:
def pad(x, max_len=64000):
    x_len = x.shape[0]
    if x_len > max_len:
        stt = np.random.randint(x_len - max_len)
        return x[stt:stt + max_len]
        # return x[:max_len]

    # num_repeats = int(max_len / x_len) + 1
    # padded_x = np.tile(x, (num_repeats))[:max_len]
    pad_length = max_len - x_len
    padded_x = np.concatenate([x, np.zeros(pad_length)], axis=0)
    return padded_x

In [16]:
class DysarthriaDetector:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model_name = 'facebook/w2v-bert-2.0'
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.model = Wav2Vec2ForSpeechClassification(model_name).to(self.device)
        # ckpt = torch.load("wave2vec2bert_wavefake.pth",map_location=self.device)
        # self.model.load_state_dict(ckpt)

        print(f"Using device: {self.device}")
        print("Dysarthria Detector initilized")

    def preprocess_audio(self, audio_path, target_sr=16000, max_length=4):
        try:
            print(f"📁 Loading audio file: {os.path.basename(audio_path)}")
            audio, sr = librosa.load(audio_path, sr=target_sr)
            original_duration = len(audio) / sr
            audio = pad(audio).reshape(-1)
            audio = audio[np.newaxis, :]
            print(f"✅ Audio loaded successfully: {original_duration:.2f}s, {sr}Hz")
            return audio, sr
        except Exception as e:
            print(f"❌ Audio processing error: {str(e)}")
            raise

    def extract_features(self, audio, sr):
        print("🔍 extract audio features...")
        feature_extractor = self.feature_extractor
        inputs = feature_extractor(audio, sampling_rate=sr, return_attention_mask=True, padding_value=0, return_tensors="pt").to(self.device)
        print("✅ Feature extraction completed")
        return inputs

    def classifier(self, features):
        model = self.model
        with torch.no_grad():
            outputs = model(**features)
            prob = outputs.logits.softmax(dim=-1)
            dysarthria_prob = prob[0][0].item()
        return dysarthria_prob

    def predict(self, audio_path):
        try:
            print("🎵 Start analyzing...")
            audio, sr = self.preprocess_audio(audio_path)
            features= self.extract_features(audio, sr)
            dysarthria_probability = self.classifier(features)
            healthy_probability = 1 - dysarthria_probability

            threshold = 0.5
            if dysarthria_probability > threshold:
                status = "DYSARTHRIA"
                prediction = "🚨 Likely dysarthric speech"
                confidence = dysarthria_probability
                color = "red"
            else:
                status = "LIKELY_HEALTHY"
                prediction = "✅ Likely normal speech"
                confidence = healthy_probability
                color = "green"

            print(f"\n{'='*50}")
            print(f"🎯 Result: {prediction}")
            print(f"📊 Confidence: {confidence:.1%}")
            print(f"📈 Real Probability: {healthy_probability:.1%}")
            print(f"📉 Fake Probability: {dysarthria_probability:.1%}")
            print(f"{'='*50}")

            duration = len(audio) / sr
            file_size = os.path.getsize(audio_path) / 1024

            result_data = {
                "status": status,
                "prediction": prediction,
                "confidence": confidence,
                "healthy_probability": healthy_probability,
                "dysarthria_probability": dysarthria_probability,
                "duration": duration,
                "sample_rate": sr,
                "file_size_kb": file_size,
            }
            return result_data

        except Exception as e:
            print(f"❌ Failed: {str(e)}")
            return {"error": str(e)}

In [17]:
detector = DysarthriaDetector()

Loading weights: 100%|██████████| 773/773 [00:00<00:00, 793.75it/s, Materializing param=masked_spec_embed]                                        


Using device: cpu
Dysarthria Detector initilized


In [15]:
def analyze_uploaded_audio(audio_file):
    if audio_file is None:
        return "Please upload audio", {}

    try:
        result = detector.predict(audio_file)

        if "error" in result:
            return f"Error: {result['error']}", {}

        status_color = "#ff4444" if result['status'] == "DYSARTHRIA" else "#44ff44"

        result_html = f"""
        <div style="padding: 20px; border-radius: 10px; background-color: {status_color}20; border: 2px solid {status_color};">
            <h3 style="color: {status_color}; margin-top: 0;">{result['prediction']}</h3>
            <p><strong>Status:</strong> {result['status']}</p>
            <p><strong>Confidence:</strong> {result['confidence']:.1%}</p>
        </div>
        """

        analysis_data = {
            "status": result['status'],
            "real_probability": f"{result['real_probability']:.1%}",
            "fake_probability": f"{result['fake_probability']:.1%}",
        }

        return result_html, analysis_data

    except Exception as e:
        error_html = f"""
        <div style="padding: 20px; border-radius: 10px; background-color: #ff444420; border: 2px solid #ff4444;">
            <h3 style="color: #ff4444;">❌ Processing error</h3>
            <p>{str(e)}</p>
        </div>
        """
        return error_html, {"error": str(e)}

In [12]:
def create_audio_interface():
    with gr.Blocks(title="Dysarthria Detector", theme=gr.themes.Soft()) as interface:
        gr.Markdown("""
        <div style="text-align: center; margin-bottom: 30px;">
            <h1 style="font-size: 28px; font-weight: bold; margin-bottom: 20px; color: #333;">
                Dysarthria Detector based on Wave2Vec2BERT speech foundation model (fine-tuned with X dataset).
            </h1>
        </div>
        <hr style="margin: 30px 0; border: none; border-top: 1px solid #e0e0e0;">
        """)

        gr.Markdown("""
        # Dysarthria Detection

        **Supported Format**: .wav, .mp3, .flac, .m4a, etc.
        """)

        with gr.Row():
            with gr.Column(scale=1):
                audio_input = gr.Audio(
                    label="📁 Upload audio file",
                    type="filepath",
                    show_label=True,
                    interactive=True
                )

                analyze_btn = gr.Button(
                    "🔍 Start analyzing",
                    variant="primary",
                    size="lg"
                )

                gr.Markdown("### 🔊 Play uploaded audio")
                audio_player = gr.Audio(
                    label="Audio Player",
                    interactive=False,
                    show_label=False
                )

            with gr.Column(scale=1):
                result_display = gr.HTML(
                    label="🎯 Results",
                    value="<p style='text-align: center; color: #666;'>Waiting for uploading...</p>"
                )

                analysis_json = gr.JSON(
                    label="📊 Detailed analysis",
                    value={}
                )

        def update_player_and_analyze(audio_file, model_type):
            if audio_file is not None:
                result_html, result_data = analyze_uploaded_audio(audio_file, model_type)
                return audio_file, result_html, result_data
            else:
                return None, "<p style='text-align: center; color: #666;'>Waiting for uploading...</p>", {}

        audio_input.change(
            fn=update_player_and_analyze,
            inputs=[audio_input],
            outputs=[audio_player, result_display, analysis_json]
        )

        analyze_btn.click(
            fn=analyze_uploaded_audio,
            inputs=[audio_input],
            outputs=[result_display, analysis_json]
        )

    return interface

In [14]:
if __name__ == "__main__":
    print("🚀 Create interface...")
    demo = create_audio_interface()

    print("📱 Launching...")
    demo.launch(
        share=False,
        debug=True,
        show_error=True
    )

🚀 Create interface...
📱 Launching...


C:\Users\Admin\AppData\Local\Temp\ipykernel_24632\1920790681.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Dysarthria Detector", theme=gr.themes.Soft()) as interface:
C:\Users\Admin\Documents\ITMOPhD\LEARNING\MachineLearning\Experiment1\.venv\Lib\site-packages\gradio\utils.py:1177: UserWarning: Expected 2 arguments for function <function create_audio_interface.<locals>.update_player_and_analyze at 0x000001C86E7EA480>, received 1.
  warnings.warn(
C:\Users\Admin\Documents\ITMOPhD\LEARNING\MachineLearning\Experiment1\.venv\Lib\site-packages\gradio\utils.py:1181: UserWarning: Expected at least 2 arguments for function <function create_audio_interface.<locals>.update_player_and_analyze at 0x000001C86E7EA480>, received 1.
  warnings.warn(
C:\Users\Admin\Documents\ITMOPhD\LEARNING\MachineLearning\Experiment1\.venv\Lib\site-packages\gradio\util

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
